# 10 — Final eğitim → model/scaler kilitleme → Eylül 2026 testi

A–D, 08 ve 09 tamamlanmış olmalı. Seçilmiş yöntem: 6 özellik / GRU /
wavelet kapalı / saf BCE. T4 kullanın; eski geliştirme ortamı korunmalı.

**Eğitim:** 18 Aralık 2025–15 Temmuz 2026 (210 gün).
**Purge:** 16 Temmuz (24 saat).
**Validation:** 17 Temmuz–30 Ağustos (45 gün).
**Test öncesi boşluk:** 31 Ağustos (24 saat).
**Test:** 1–30 Eylül 2026, UTC; 720 tahmin, seed 42/43/44 ayrı.

Üç final model yalnızca train'de öğrenir, epoch'u validation BCE seçer.
Train+validation ile yeniden fit yapılmaz. Sonraki hücre bütün model/scaler
hashlerini kilitler. **Test kaynağı yalnızca bu kilit oluşturulduktan sonra
indirilir.** Testte eğitim, model/seed/eşik seçimi veya ağırlık güncelleme yok.

Her adımın checkpoint/durum/ZIP'i Drive'da. Kesinti sonrası aynı commit,
ayar ve ortamla baştan çalıştırın. Bitmiş adımlar/seeds tekrar eğitilmez.
İki runtime aynı çıktı alanına yazmasın. Bu tamamlanmış dönem üzerinde
geriye dönük dokunulmamış holdout'tur; ileriye dönük deney iddiası değildir.

In [ ]:
import os, sys, subprocess, json, shutil, hashlib
from pathlib import Path
from datetime import datetime, timezone
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ['ADVISOR_LOCK_ROOT'] = '/content/advisor_runtime/locks'
import torch
assert torch.cuda.is_available(), 'Runtime → Change runtime type → GPU seçin.'
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)

# İlk aşamada yalnızca A. Sonuçlara bakarak seed/fold veya ölçüt değiştirmeyin.
EXPERIMENTS = ['FINAL']
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_BRANCH = 'codex/advisor-ablation-protocol'
REPO_COMMIT = ''  # Boş: dalın güncel commit'i. Devam için eski commit'i buraya sabitleyebilirsiniz.
REPO_DIR = Path('/content/yenibot_advisor_repo')
DRIVE_BASE = Path('/content/drive/MyDrive/yeniBot')
LOCAL_BASE = Path('/content/advisor_runtime')
AUTO_UNASSIGN = False  # True: başarıyla eğitim + ZIP tamamlandıktan sonra GPU oturumunu bırakır.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PERSIST = DRIVE_BASE / 'advisor_experiments' / 'colab_v1'
for folder in [PERSIST / 'inputs', PERSIST / 'data', PERSIST / 'runs', PERSIST / 'sessions', PERSIST / 'reports', LOCAL_BASE]:
    folder.mkdir(parents=True, exist_ok=True)
print('Kalıcı çıktı alanı:', PERSIST)

## Git'ten güncel deney kodunu çek

Doğru dal açıkça seçilir; commit ekrana ve sonuç paketine kaydedilir. Özel depo için Git kimlik
doğrulaması gerekirse mevcut hesabınızla erişim sağlayın; notebooka token yazmayın.
Çekilen kodu eğitim ayrı Python işleminde yükler. Önceki hücrelerden `yenibot` modülleri
yüklenmişse kodu güncellemeden önce oturumu yeniden başlatmanız istenir.

In [ ]:
def git(*args):
    return subprocess.check_output(['git', '-C', str(REPO_DIR), *args], text=True).strip()

if any(name == 'yenibot' or name.startswith('yenibot.') for name in sys.modules):
    raise RuntimeError('Kod hücresini tekrar çalıştırmadan önce Runtime → Restart session; sonra baştan çalıştırın.')
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    assert not git('status', '--porcelain', '--untracked-files=no'), 'Colab çalışma kopyasında değişiklik var; otomatik silinmez.'
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    if REPO_COMMIT:
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_COMMIT], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'merge', '--ff-only', 'origin/' + REPO_BRANCH], check=True)
if REPO_COMMIT:
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_COMMIT], check=True)
repo_commit = git('rev-parse', 'HEAD')
assert (REPO_DIR / 'yenibot/training/advisor.py').exists(), 'Yanlış dal veya eksik deney kodu.'
print('Dal:', REPO_BRANCH, '| Commit:', repo_commit)
sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

In [ ]:
# Colab'ın mevcut GPU Torch paketini zorla değiştirmeyin; upgrade kullanılmıyor.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements.txt')], check=True)
import yaml
from yenibot.training.advisor import sha256, atomic_json
from yenibot.training.advisor_colab import (freeze_market_inputs, cache_frozen_inputs, create_review_bundle,
                                          freeze_oi_inputs, pin_completed_a_reference)
from yenibot.data import download_full_klines
from yenibot.data.binance import download_futures_metrics_from_vision
session_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
SESSION = PERSIST / 'sessions' / session_stamp
SESSION.mkdir(parents=True, exist_ok=True)
packages = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(SESSION / 'packages.txt').write_text(packages)
session_meta = {'session': session_stamp, 'branch': REPO_BRANCH, 'commit': repo_commit,
                'gpu': torch.cuda.get_device_name(0), 'python': sys.version, 'torch': torch.__version__,
                'experiments': EXPERIMENTS, 'status': 'setup', 'test_evaluations': 0}
atomic_json(SESSION / 'session.json', session_meta)
print('Ortam ve kaynak kaydı:', SESSION)

## Yöntem kararını ve final takvimini sabitle

Aynı `colab_v1` Drive alanından A–D, wavelet ve loss kararları denetlenir.
Final çıktılar ayrı `final_test_v1/` alanına kaydedilir; geliştirme kayıtları
değişmez. Saatlik temel özellikler için 4H/OI kaynağı indirilmez.

In [ ]:
import pandas as pd
from yenibot.training.advisor_final import selected_contract
cfg=yaml.safe_load((REPO_DIR/'configs/advisor_final_execution.yaml').read_text())
cfg['reservation_config']=str(REPO_DIR/'configs/advisor_final_test.yaml')
cfg['stage_config']=str(REPO_DIR/'configs/advisor_stages.yaml')
cfg['development_root']=str(PERSIST)
cfg['raw']['base_snapshot']=str(PERSIST/'inputs/market_snapshot_v1')
FINAL_ROOT=PERSIST/'final_test_v1'
cfg['output']=str(FINAL_ROOT)
runtime_config=LOCAL_BASE/'advisor_final.yaml'
runtime_config.write_text(yaml.safe_dump(cfg,sort_keys=False))
SESSION=FINAL_ROOT/'sessions'/session_stamp
SESSION.mkdir(parents=True,exist_ok=True)
(SESSION/'packages.txt').write_text(packages)
shutil.copyfile(runtime_config,SESSION/runtime_config.name)
session_meta.update(phase='final',status='plan_frozen')
atomic_json(SESSION/'session.json',session_meta)
contract=selected_contract(cfg)
print('Yöntem:',cfg['selected_method'])
print('Eğitim/doğrulama:',cfg['fit'])
print('Test rezervasyonu:',contract['reservation']['test'])
print('Final çıktı:',FINAL_ROOT)

In [ ]:
def execute_logged(command, log_name):
    # Ayrı süreç güncel kodu yükler; hata kodu başarısızlığı gizlemez.
    with (SESSION / log_name).open('a', buffering=1) as log:
        with subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                return_code = process.wait()
            except BaseException:
                process.terminate()
                process.wait()
                raise
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

def execute_final(action):
    session_meta.update(status='running',current_action=action)
    atomic_json(SESSION/'session.json',session_meta)
    try:
        execute_logged([sys.executable,'-u','-m','yenibot.training.advisor_final',action,
                        '--config',str(runtime_config)], action+'.log')
        session_meta.update(status='step_complete',last_completed_action=action)
        session_meta.pop('error',None)
    except BaseException as exc:
        session_meta.update(status='interrupted_or_failed',error=repr(exc))
        raise
    finally:
        session_meta['updated_at']=datetime.now(timezone.utc).isoformat()
        atomic_json(SESSION/'session.json',session_meta)
        subprocess.run([sys.executable,'-m','yenibot.training.advisor_final','bundle',
                        '--config',str(runtime_config)],check=True)

## 1 — Test öncesi saatlik kaynağı ve girdileri hazırla

2022 başlangıcındaki sabit ham geçmiş korunur, eksik devam aylık cache ile yalnızca 31 Ağustos 09:00 UTC barına kadar indirilir. Eylül kaynağı okunmaz. Altı özellik eski saatlik hesaplayıcıyla üretilir.

In [ ]:
execute_final('prepare-fit')

## 2 — Üç final modeli eğit

5.040 train satırı → 4.977 dizi; 1.080 validation satırı → 1.017 dizi. Epoch seçimi normal validation BCE. Her epoch Drive checkpoint; seedler 42/43/44. Test sonucu yok.

In [ ]:
execute_final('train')

## 3 — Model, scaler ve ayar hashlerini kilitle

Üç seed tamamlanmadan kilit oluşturulmaz. Her modelin en iyi validation epoch ağırlığı ve sadece train ile fit edilen scaler saklanır. Test verisini indirmek için bu manifest zorunludur.

In [ ]:
execute_final('freeze')

## 4 — Kilitten sonra Eylül test kaynağını hazırla

İleri getiriler için son gereken saat 1 Ekim 09:00 UTC. İlk tahmin için 63 geçmiş gözlem eklenir: tablo 783 satır, tahmin sayısı 720. Fit geçmişi yeniden hesaplandığında birebir aynı olmalı.

In [ ]:
execute_final('prepare-test')

## 5 — Donmuş modellerle final testi değerlendir

Testte fit yok. Her seed ayrı 720 tahmin; BCE/AP/F1/precision/recall/accuracy/Rank IC ve mean/std. Mean/std seed kararlılığıdır, bağımsız örnek güven aralığı değildir. Sabit train-sınıf-frekansı ve hep-negatif referansları da raporlanır.

In [ ]:
execute_final('evaluate-test')

In [ ]:
test_results=json.loads((FINAL_ROOT/'runs/test/test_results.json').read_text())
session_meta.update(status='complete',test_evaluations=1)
atomic_json(SESSION/'session.json',session_meta)
subprocess.run([sys.executable,'-m','yenibot.training.advisor_final','bundle',
                '--config',str(runtime_config)],check=True)
display(pd.DataFrame(test_results['per_seed']))
display(pd.DataFrame(test_results['aggregate_mean_std']).T)
print('Basit referanslar:',test_results['baselines'])
print('İnceleme ZIP:',FINAL_ROOT/'reports/advisor_final_latest_review_bundle.zip')
if AUTO_UNASSIGN:
    from google.colab import runtime
    runtime.unassign()

## İnceleme için getirilecek ZIP

`MyDrive/yeniBot/advisor_experiments/colab_v1/final_test_v1/reports/advisor_final_latest_review_bundle.zip`

ZIP: final plan, kaynak/kilit hashleri, validation ve test tahminleri,
metrikler, logs ve durumlar. Ağırlıklar ve büyük ham veri ZIP'e girmez.
Başarısız hücrede de kısmi ZIP hazırlanır; son checkpoint Drive'da kalır.
Test tamamlandıktan sonra aynı notebook aynı kimlikte raporu yeniden kullanır.
Test sonuçlarına göre modeli/eşiği değiştirmek bu holdout'u geliştirme verisine
dönüştürür; böyle bir değişiklik bu protokolün kapsamında değildir.